In [1]:
# --- Step 1: 提取 SCL / SCR ---
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt

INPUT_FILE = 'GSR_SCL_SCR_AllEvents.xlsx'
FS = 4  # 采样率 Hz
CUTOFF = 0.1  # SCL 低通截止频率 Hz
ORDER = 4  # 巴特沃斯阶数


# --- 工具函数 ---
def compute_scl(signal, fs=FS, cutoff=CUTOFF, order=ORDER):
    padlen = 3 * (order + 1)
    if len(signal) > padlen:
        nyq = 0.5 * fs
        normal_cutoff = cutoff / nyq
        b, a = butter(order, normal_cutoff, btype='low')
        y = filtfilt(b, a, signal)
    else:
        # 短信号 → 滑动平均
        window_size = max(3, len(signal) // 2)
        y = pd.Series(signal).rolling(window=window_size, min_periods=1, center=True).mean().values
    return y


# --- 读取数据 ---
df = pd.read_excel(INPUT_FILE)
group_cols = ['阶段', '组别', '姓名', '飞行天数']

scl_rows, scr_rows = [], []

# --- 遍历分组 ---
for keys, group in df.groupby(group_cols):
    gsr_signal = group['data'].values.astype(float)
    scl_signal = compute_scl(gsr_signal, fs=FS, cutoff=CUTOFF, order=ORDER)
    scr_signal = gsr_signal - scl_signal

    scl_rows.append(group.assign(data=scl_signal))
    scr_rows.append(group.assign(data=scr_signal))

# --- 合并并保存 ---
scl_all = pd.concat(scl_rows, ignore_index=True)
scr_all = pd.concat(scr_rows, ignore_index=True)

scl_all.to_excel('GSR_SCL_AllEvents.xlsx', index=False)
scr_all.to_excel('GSR_SCR_AllEvents.xlsx', index=False)

print("✅ Step 1 完成：SCL / SCR 文件已生成")


✅ Step 1 完成：SCL / SCR 文件已生成


In [2]:
# --- Step 2: 基线校准 + 3SD 异常值剔除 ---
import pandas as pd
import numpy as np

FS = 4
PRE_SECONDS = 1.0
MIN_PRE_POINTS = 1


# --- 基线校准函数 ---
def apply_baseline(df, orig_df, pre_seconds=PRE_SECONDS, fs=FS, min_pre_points=MIN_PRE_POINTS):
    df = df.reset_index(drop=False).rename(columns={'index': 'orig_index'})
    orig_df = orig_df.reset_index(drop=True)

    adjusted_rows = []
    group_cols = ['阶段', '组别', '姓名', '飞行天数']

    for keys, group in df.groupby(group_cols):
        phase, grp, name, day = keys
        gsr_signal = group['data'].values.astype(float)
        start_idx = int(group['orig_index'].iloc[0])

        pre_pts = int(round(pre_seconds * fs))
        pre_start = max(0, start_idx - pre_pts)
        pre_end = start_idx
        if pre_end - pre_start >= min_pre_points:
            pre_vals = orig_df.loc[pre_start:pre_end - 1, 'data'].values.astype(float)
            baseline = np.mean(pre_vals)
        else:
            hejiu_vals = orig_df[(orig_df['阶段'] == 'hejiu') &
                                 (orig_df['姓名'] == name) &
                                 (orig_df['飞行天数'] == day)]['data'].values.astype(float)
            baseline = np.mean(hejiu_vals) if len(hejiu_vals) > 0 else 0.0

        gsr_baselined = gsr_signal - baseline
        adjusted_rows.append(group.assign(data=gsr_baselined))

    return pd.concat(adjusted_rows, ignore_index=True)


# --- 读取原始文件和 Step1 文件 ---
orig_df = pd.read_excel('GSR_SCL_SCR_AllEvents.xlsx')
scl_df = pd.read_excel('GSR_SCL_AllEvents.xlsx')
scr_df = pd.read_excel('GSR_SCR_AllEvents.xlsx')

# --- 基线校准 ---
scl_adj = apply_baseline(scl_df, orig_df)
scr_adj = apply_baseline(scr_df, orig_df)

# 去除缺失值
scl_adj = scl_adj.dropna().reset_index(drop=True)
scr_adj = scr_adj.dropna().reset_index(drop=True)

# --- 3SD 异常值筛选 ---
all_stats = []
outliers_detail = []
scl_cleaned_rows = []

for (name, day), sub_df in scl_adj.groupby(['姓名', '飞行天数']):
    mean_val = sub_df['data'].mean()
    std_val = sub_df['data'].std()

    df_high = sub_df[sub_df['data'] > mean_val + 3 * std_val]
    df_low = sub_df[sub_df['data'] < mean_val - 3 * std_val]

    total = len(sub_df)
    high_count = len(df_high)
    low_count = len(df_low)

    # 统计信息
    all_stats.append({
        '姓名': name,
        '飞行天数': day,
        '当天飞行数据总数': total,
        '高于+3SD数量': high_count,
        '高于+3SD比例': high_count / total if total > 0 else 0,
        '低于-3SD数量': low_count,
        '低于-3SD比例': low_count / total if total > 0 else 0,
        '去除率': (high_count + low_count) / total if total > 0 else 0
    })

    # 异常值详情
    for idx, row in df_high.iterrows():
        outliers_detail.append({
            '姓名': name,
            '飞行天数': day,
            '异常类型': '高于+3SD',
            'SCL': row['data'],
            '均值': mean_val,
            '标准差': std_val,
            '偏离程度(SD)': (row['data'] - mean_val) / std_val,
            '原始数据索引': int(row['orig_index'])
        })
    for idx, row in df_low.iterrows():
        outliers_detail.append({
            '姓名': name,
            '飞行天数': day,
            '异常类型': '低于-3SD',
            'SCL': row['data'],
            '均值': mean_val,
            '标准差': std_val,
            '偏离程度(SD)': (row['data'] - mean_val) / std_val,
            '原始数据索引': int(row['orig_index'])
        })

    # 剔除异常值后的数据
    cleaned = sub_df[~sub_df.index.isin(df_high.index.union(df_low.index))]
    scl_cleaned_rows.append(cleaned)

# 合并
scl_cleaned = pd.concat(scl_cleaned_rows, ignore_index=True)

# --- 保存结果 ---
# 1) 校准后的原始数据（未剔除异常值）
scl_adj.to_excel('GSR_SCL_AllEvents_pre1sBaseline.xlsx', index=False)
scr_adj.to_excel('GSR_SCR_AllEvents_pre1sBaseline.xlsx', index=False)

# 2) 剔除异常值后的数据
scl_cleaned.to_excel('GSR_SCL_AllEvents_pre1sBaseline_3sd.xlsx', index=False)

# 3) 异常值统计表 & 详情表
pd.DataFrame(all_stats).to_excel('GSR_SCL_被试分天异常值统计.xlsx', index=False)
pd.DataFrame(outliers_detail).to_excel('GSR_SCL_被试分天异常值详情.xlsx', index=False)

# 4) 同步清理原始 scl 文件，方便画图
orig_scl = pd.read_excel('GSR_SCL_AllEvents.xlsx')
orig_scl_cleaned = orig_scl[~orig_scl.index.isin([o['原始数据索引'] for o in outliers_detail])]
orig_scl_cleaned.to_excel('GSR_SCL_AllEvents_cleanForPlot.xlsx', index=False)

print("✅ Step 2 完成：基线校准 + 缺失值剔除 + 3SD异常值检测 & 输出统计表")


✅ Step 2 完成：基线校准 + 缺失值剔除 + 3SD异常值检测 & 输出统计表


In [3]:
import pandas as pd
import numpy as np

# 读取数据
scl_adj = pd.read_excel('GSR_SCL_AllEvents_pre1sBaseline.xlsx')  # 校准后的数据
scl_cleaned = pd.read_excel('GSR_SCL_AllEvents_pre1sBaseline_3sd.xlsx')  # 剔除异常值后的数据

# --- 1. 缺失值统计 ---
total_points = len(scl_adj)
missing_points = scl_adj['data'].isna().sum()
missing_ratio = missing_points / total_points

print(f"全体数据总数: {total_points}")
print(f"缺失值数量: {missing_points}")
print(f"缺失值占比: {missing_ratio:.2%}")

# --- 2. 异常值统计（±3SD） ---
# 异常值数 = 校准数据数量 - 剔除异常值后的数量
cleaned_points = len(scl_cleaned)
outlier_points = total_points - missing_points - cleaned_points
outlier_ratio = outlier_points / total_points

print(f"异常值数量（±3SD）: {outlier_points}")
print(f"异常值占比: {outlier_ratio:.2%}")

# --- 3. 可选: 全体有效数据比例 ---
valid_points = total_points - missing_points - outlier_points
valid_ratio = valid_points / total_points
print(f"有效数据占比: {valid_ratio:.2%}")


全体数据总数: 405106
缺失值数量: 0
缺失值占比: 0.00%
异常值数量（±3SD）: 5640
异常值占比: 1.39%
有效数据占比: 98.61%


In [6]:
# --- Step 3: SCR 峰值筛选 ---
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

FS = 4
MIN_AMP = 0.02
MIN_INTERVAL_S = 0.25
RISE_TIME_MIN = 0.5
RISE_TIME_MAX = 3.0

from scipy.signal import find_peaks


def find_scr_peaks(scr_signal, fs=FS, min_amp=MIN_AMP, min_interval_s=MIN_INTERVAL_S,
                   rise_min_s=RISE_TIME_MIN, rise_max_s=RISE_TIME_MAX,
                   compute_features=True, auto_adjust_rise=True):
    """
    自适应 SCR 峰值检测
    - scr_signal: 信号数组
    - fs: 采样率
    - min_amp: 峰最小幅度
    - min_interval_s: 峰最小间隔（秒）
    - rise_min_s: 上升时间最小值（秒）
    - rise_max_s: 上升时间最大值（秒）
    - compute_features: 是否计算上升时间和半恢复时间
    - auto_adjust_rise: 是否根据采样率自动调整上升时间范围
    """
    distance = int(round(min_interval_s * fs))
    peaks, _ = find_peaks(scr_signal, height=min_amp, distance=distance)

    if not compute_features:
        return peaks

    # 根据采样率自动调整上升时间范围
    if auto_adjust_rise:
        # 至少1个采样点，最多峰宽的一半
        rise_min_pts = max(1, int(round(rise_min_s * fs)))
        rise_max_pts = max(rise_min_pts, int(round(rise_max_s * fs)))
    else:
        rise_min_pts = int(round(rise_min_s * fs))
        rise_max_pts = int(round(rise_max_s * fs))

    results = []
    for p in peaks:
        amp = scr_signal[p]

        # 上升时间（从峰前最低点到峰顶）
        left = p
        while left > 0 and scr_signal[left] > scr_signal[left - 1]:
            left -= 1
        rise_time_pts = p - left

        # 判断是否在允许的上升时间范围
        if rise_time_pts < rise_min_pts:
            rise_time_pts = rise_min_pts  # 调整为最小值，避免丢峰
        elif rise_time_pts > rise_max_pts:
            rise_time_pts = rise_max_pts  # 调整为最大值

        rise_time_s = rise_time_pts / fs

        # 半恢复时间（从峰顶下降到一半幅值）
        half_amp = amp / 2.0
        right = p
        while right < len(scr_signal) - 1 and scr_signal[right] > half_amp:
            right += 1
        half_recovery_s = (right - p) / fs

        results.append({
            'peak_idx_within_segment': int(p),
            '峰振幅': float(amp),
            '上升时间_s': float(rise_time_s),
            '半恢复时间_s': float(half_recovery_s)
        })

    return results


# --- 读取 SCR ---
scr_df = pd.read_excel('GSR_SCR_AllEvents_pre1sBaseline.xlsx')
group_cols = ['阶段', '组别', '姓名', '飞行天数']

peaks_list = []
for keys, group in scr_df.groupby(group_cols):
    scr_signal = group['data'].values.astype(float)
    peaks = find_scr_peaks(scr_signal, fs=FS)
    for p in peaks:
        peaks_list.append({
            '阶段': keys[0],
            '组别': keys[1],
            '姓名': keys[2],
            '飞行天数': keys[3],
            **p
        })

peaks_df = pd.DataFrame(peaks_list)
peaks_df.to_excel('GSR_SCR_Peaks_pre1sBaseline.xlsx', index=False)

print("✅ Step 3 完成：峰值筛选文件已生成")


✅ Step 3 完成：峰值筛选文件已生成
